In [ ]:
import numpy as np 
import pandas as pd
pd.options.display.max_columns = 50
import matplotlib.pyplot as plt
import seaborn as sns

import os
import gc
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# LGBM

In [ ]:
path = '/datas/zhangyy/5440/'

sales = pd.read_csv(os.path.join(path, "sales_train_evaluation.csv"))
calendar = pd.read_csv(os.path.join(path, "calendar.csv"))
prices = pd.read_csv(os.path.join(path, "sell_prices.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))


for d in range(1942,1970):
    col = 'd_' + str(d)
    sales[col] = 0
    sales[col] = sales[col].astype(np.int16)

In [ ]:
def downcast(df):
    cols = df.dtypes.index.tolist() 
    types = df.dtypes.values.tolist()
    for i,t in enumerate(types):
        if 'int' in str(t):
            if df[cols[i]].min() > np.iinfo(np.int8).min and df[cols[i]].max() < np.iinfo(np.int8).max:
                df[cols[i]] = df[cols[i]].astype(np.int8)
            elif df[cols[i]].min() > np.iinfo(np.int16).min and df[cols[i]].max() < np.iinfo(np.int16).max:
                df[cols[i]] = df[cols[i]].astype(np.int16)
            elif df[cols[i]].min() > np.iinfo(np.int32).min and df[cols[i]].max() < np.iinfo(np.int32).max:
                df[cols[i]] = df[cols[i]].astype(np.int32)
            else:
                df[cols[i]] = df[cols[i]].astype(np.int64)
        elif 'float' in str(t):
            if df[cols[i]].min() > np.finfo(np.float16).min and df[cols[i]].max() < np.finfo(np.float16).max:
                df[cols[i]] = df[cols[i]].astype(np.float16)
            elif df[cols[i]].min() > np.finfo(np.float32).min and df[cols[i]].max() < np.finfo(np.float32).max:
                df[cols[i]] = df[cols[i]].astype(np.float32)
            else:
                df[cols[i]] = df[cols[i]].astype(np.float64)
        elif t == object:
            if cols[i] == 'date':
                df[cols[i]] = pd.to_datetime(df[cols[i]], format='%Y-%m-%d')
            else:
                df[cols[i]] = df[cols[i]].astype('category')
    return df  

sales = downcast(sales)
prices = downcast(prices)
calendar = downcast(calendar)

df = pd.melt(sales, id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], var_name='d', value_name='sold').dropna()
df = pd.merge(df, calendar, on='d', how='left')
df = pd.merge(df, prices, on=['store_id','item_id','wm_yr_wk'], how='left') 
display(df.head())
print(df.info())

In [ ]:
d_id = dict(zip(df.id.cat.codes, df.id))
d_store_id = dict(zip(df.store_id.cat.codes, df.store_id))

df.drop(["date", "wm_yr_wk", "weekday"],axis=1,inplace=True)

df.d = df['d'].apply(lambda x: x.split('_')[1]).astype(np.int16)

In [ ]:
import category_encoders as ce
# lag特征的追加 (Lag Features)

lags = [1, 2, 3, 7, 14, 28, 56]
for lag in lags:
    df['sold_lag_' + str(lag)] = df.groupby(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'], as_index=False)['sold'].shift(lag + 28).astype(np.float16)

# Rolling Mean Features
sold_lag_cols = ['sold_lag_7', 'sold_lag_28']
for win in [7, 28]:
    for lag, lag_col in zip([7, 28], sold_lag_cols):
    
        if 'sold_lag_' + str(lag) in df.columns:
            df['rmean_{}_{}'.format(lag, win)] = df.groupby(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])['sold_lag_' + str(lag)].transform(lambda x: x.rolling(window=win).mean()).astype(np.float16)

# Target Encoding
print("执行目标编码前，'state_id' 的数据类型:", df['state_id'].dtype)
cols_to_target_encode = ['state_id', 'store_id', 'cat_id', 'dept_id']
new_cols = ['state_id_ots', 'store_id_ots', 'cat_id_ots', 'dept_id_ots']

existing_cols = [col for col in cols_to_target_encode if col in df.columns]
existing_new_cols = [nc for c, nc in zip(cols_to_target_encode, new_cols) if c in existing_cols]

if existing_cols:
    te = ce.CatBoostEncoder(random_state=42)
    df[existing_new_cols] = te.fit_transform(df[existing_cols], df['sold'])



cols = df.columns
for col in cols:
    if df[col].dtype.name == 'category':
        print(f"  正在编码列: {col}")
        df[col] = df[col].cat.codes

print("执行数字编码后，'state_id' 的数据类型:", df['state_id'].dtype)

df = downcast(df)
min_day = 57
if df.shape[0] >= min_day:
     df = df.iloc[min_day-1:].copy()
else:
     df = df.iloc[0:0].copy()

df.info()

In [ ]:
df

In [ ]:
df.to_pickle('/home/zhangyy/MAFS5440/data.pkl')
del df
gc.collect();

In [ ]:
data = pd.read_pickle('/home/zhangyy/MAFS5440/data.pkl')
valid = data[(data['d']>=1914) & (data['d']<1942)][['id','d','sold']]
test = data[data['d']>=1942][['id','d','sold']]
eval_preds = test['sold']
valid_preds = valid['sold']

In [ ]:
valid = data[(data['d']>=1914) & (data['d']<1942)][['id','d','sold']]
test = data[data['d']>=1942][['id','d','sold']]
eval_preds = test['sold']
valid_preds = valid['sold']

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import joblib
import optuna
import pandas as pd
import numpy as np
import gc
import sklearn.metrics

# 定义特征和处理数据类型
base_features = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday',
    'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2',
    'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price',
    'sold_lag_1', 'sold_lag_2', 'sold_lag_3', 'sold_lag_7', 'sold_lag_14',
    'sold_lag_28', 'sold_lag_56', 'rmean_7_7', 'rmean_28_7', 'rmean_7_28',
    'rmean_28_28', 'state_id_ots', 'store_id_ots', 'cat_id_ots', 'dept_id_ots'
]

categorical_features = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday',
    'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2'
]

for col in categorical_features:
    if col in data.columns:
        data[col] = data[col].astype('category')

# 划分store_id
d_store_id = {i: f'STORE_{i}' for i in data['store_id'].cat.codes.unique()}
stores = data['store_id'].cat.codes.unique().tolist()
data['store_id_code'] = data['store_id'].cat.codes


data['predictions'] = 0.0
seed=42

X_train, y_train, X_valid, y_valid = [None] * 4

def objectives(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'rmse',
        'n_estimators': 1000,
        'verbosity': -1,
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth' : trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }

    model = LGBMRegressor(random_state=seed, **params)

    model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              eval_metric='rmse',
              callbacks=[early_stopping(stopping_rounds=20, verbose=False)])

    preds = model.predict(X_valid)
    score = np.sqrt(sklearn.metrics.mean_squared_error(y_valid, preds))

    return score

for store_code in stores:
    global X_train, y_train, X_valid, y_valid

    df_store = data[data['store_id_code'] == store_code]

    features_for_store = [f for f in base_features if f != 'store_id']

    train_mask = df_store['d'] < 1914
    valid_mask = (df_store['d'] >= 1914) & (df_store['d'] < 1942)
    test_mask = df_store['d'] >= 1942

    X_train = df_store.loc[train_mask, features_for_store]
    y_train = df_store.loc[train_mask, 'sold']

    X_valid = df_store.loc[valid_mask, features_for_store]
    y_valid = df_store.loc[valid_mask, 'sold']

    X_test = df_store.loc[test_mask, features_for_store]

    if X_train.empty or X_valid.empty:
        print(f"Skipping store {d_store_id[store_code]} due to empty training/validation set.")
        continue

    store_name = d_store_id[store_code]
    print(f'\n***** Training and Prediction for Store: {store_name} *****')

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objectives, n_trials=10) 
    best_params = study.best_params
    best_params['random_state'] = seed
    best_params['n_estimators'] = 2000
    best_params['objective'] = 'regression_l1'
    best_params['metric'] = 'rmse'

    print(f"Best trial for store {store_name}: RMSE = {study.best_trial.value}")
    print(f"Best params: {best_params}")

    model = LGBMRegressor(**best_params)

    model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              eval_metric='rmse',
              callbacks=[early_stopping(stopping_rounds=50), log_evaluation(period=100)])

    valid_indices = X_valid.index
    test_indices = X_test.index

    if not X_valid.empty:
        data.loc[valid_indices, 'predictions'] = model.predict(X_valid)
    if not X_test.empty:
        data.loc[test_indices, 'predictions'] = model.predict(X_test)

    filename = f'model_{store_name}.pkl'
    joblib.dump(model, filename)
    print(f"Model for {store_name} saved as {filename}")

    del model, X_train, y_train, X_valid, y_valid, X_test, df_store
    gc.collect()


print(data.loc[data['d'] >= 1914, ['id', 'sold', 'predictions']].head())


In [ ]:
# 生成提交文件 

if 'id' not in data.columns:
    data.reset_index(inplace=True)
    if 'index' in data.columns and 'id' not in data.columns:
        data.rename(columns={'index': 'id'}, inplace=True)


# 筛选出验证集周期的预测
validation_df = data[(data['d'] >= 1914) & (data['d'] < 1942)][['id', 'd', 'predictions']]
validation_df = pd.pivot(validation_df, index='id', columns='d', values='predictions').reset_index()
validation_df.columns = ['id'] + ['F' + str(i + 1) for i in range(28)]
# 转换'id'列
validation_df['id'] = validation_df['id'].astype(str).str.replace('_evaluation', '_validation')

# 筛选出测试集（评估）周期的预测
evaluation_df = data[data['d'] >= 1942][['id', 'd', 'predictions']]
evaluation_df = pd.pivot(evaluation_df, index='id', columns='d', values='predictions').reset_index()
evaluation_df.columns = ['id'] + ['F' + str(i + 1) for i in range(28)]

# 合并验证集和评估集的预测
submit = pd.concat([validation_df, evaluation_df]).reset_index(drop=True)

# 保存到 CSV
submit.to_csv('submission.csv', index=False)

print(submit.head())

# NN model

In [ ]:
import pandas as pd
import numpy as np
import gc
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 检查是否有可用的 GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

base_features = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday',
    'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2',
    'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price',
    'sold_lag_1', 'sold_lag_2', 'sold_lag_3', 'sold_lag_7', 'sold_lag_14',
    'sold_lag_28', 'sold_lag_56', 'rmean_7_7', 'rmean_28_7', 'rmean_7_28',
    'rmean_28_28', 'state_id_ots', 'store_id_ots', 'cat_id_ots', 'dept_id_ots'
]

categorical_features = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday',
    'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2'
]



existing_cat_features = [col for col in categorical_features if col in data.columns]
for col in existing_cat_features:
    data[col] = data[col].astype('category')


numerical_features = [f for f in base_features if f not in categorical_features and f in data.columns]

# 按 store_id 划分
d_store_id = {i: f'STORE_{i}' for i in data['store_id'].cat.codes.unique()}
stores = data['store_id'].cat.codes.unique().tolist()
data['store_id_code'] = data['store_id'].cat.codes

data['predictions'] = 0.0
seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

class M5Dataset(Dataset):
    """自定义PyTorch数据集"""
    def __init__(self, df, cat_features, num_features, target_col='sold'):
        self.df = df
        self.cat_features = cat_features
        self.num_features = num_features
        self.target_col = target_col

        self.cats = [df[col].cat.codes.values for col in self.cat_features]
        self.nums = df[self.num_features].values
        
        if self.target_col in df.columns:
            self.targets = df[self.target_col].values
        else:
            self.targets = np.zeros(len(df)) # For test set

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        cat_values = [torch.tensor(self.cats[i][idx], dtype=torch.long) for i in range(len(self.cat_features))]
        num_values = torch.tensor(self.nums[idx], dtype=torch.float)
        target = torch.tensor(self.targets[idx], dtype=torch.float)
        
        return cat_values, num_values, target


class M5Model(nn.Module):
    """包含Embedding层的神经网络模型"""
    def __init__(self, embedding_sizes, n_numeric):
        super().__init__()
        # 创建Embedding层
        self.embeddings = nn.ModuleList([nn.Embedding(categories, size) for categories, size in embedding_sizes])
        n_emb = sum(e.embedding_dim for e in self.embeddings)
        self.n_emb, self.n_numeric = n_emb, n_numeric
        
        # 线性层
        self.lin1 = nn.Linear(self.n_emb + self.n_numeric, 512)
        self.lin2 = nn.Linear(512, 256)
        self.lin3 = nn.Linear(256, 1)
        
        # Batch Normalization 和 Dropout
        self.bn1 = nn.BatchNorm1d(self.n_numeric)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(256)
        self.emb_drop = nn.Dropout(0.25)
        self.drops = nn.Dropout(0.5)
        
        # 激活函数
        self.activation = nn.ReLU()

    def forward(self, x_cat, x_num):
        # 处理Embedding层
        x = [e(x_cat[i]) for i, e in enumerate(self.embeddings)]
        x = torch.cat(x, 1)
        x = self.emb_drop(x)
        
        # 处理数值特征
        x_num = self.bn1(x_num)
        
        # 拼接两种特征
        x = torch.cat([x, x_num], 1)

        x = self.activation(self.lin1(x))
        x = self.drops(self.bn2(x))
        x = self.activation(self.lin2(x))
        x = self.drops(self.bn3(x))
        x = self.lin3(x)
        
        return x


# 定义超参数
EPOCHS = 20
BATCH_SIZE = 1024
LEARNING_RATE = 0.001
PATIENCE = 5 # 早停

for store_code in stores:
    store_name = d_store_id[store_code]
    print(f'\n***** Training and Prediction for Store: {store_name} *****')
    
    # 划分数据
    df_store = data[data['store_id_code'] == store_code]
    features_for_store = [f for f in base_features if f != 'store_id']
    
    cat_features_store = [f for f in existing_cat_features if f != 'store_id']
    num_features_store = [f for f in numerical_features if f != 'store_id']
    
    train_mask = df_store['d'] < 1914
    valid_mask = (df_store['d'] >= 1914) & (df_store['d'] < 1942)
    test_mask = df_store['d'] >= 1942
    
    train_df = df_store[train_mask].copy()
    valid_df = df_store[valid_mask].copy()
    test_df = df_store[test_mask].copy()

    if train_df.empty or valid_df.empty:
        print(f"Skipping store {store_name} due to empty training/validation set.")
        continue
    
    for col in num_features_store:
        train_df[col] = train_df[col].astype(np.float32)
        valid_df[col] = valid_df[col].astype(np.float32)
        if not test_df.empty:
            test_df[col] = test_df[col].astype(np.float32)
            
    # 标准化数值Feature
    scaler = StandardScaler()
    train_df.loc[:, num_features_store] = scaler.fit_transform(train_df[num_features_store])
    valid_df.loc[:, num_features_store] = scaler.transform(valid_df[num_features_store])
    if not test_df.empty:
        test_df.loc[:, num_features_store] = scaler.transform(test_df[num_features_store])

    train_df[num_features_store] = train_df[num_features_store].fillna(0)
    valid_df[num_features_store] = valid_df[num_features_store].fillna(0)
    if not test_df.empty:
        test_df[num_features_store] = test_df[num_features_store].fillna(0)

    # 创建数据集和数据加载器
    train_dataset = M5Dataset(train_df, cat_features_store, num_features_store)
    valid_dataset = M5Dataset(valid_df, cat_features_store, num_features_store)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    # 定义模型
    # 获取每个分类特征的类别数
    cat_dims = [len(df_store[col].cat.categories) for col in cat_features_store]
    emb_dims = [(x, min(50, (x + 1) // 2)) for x in cat_dims]
    
    model = M5Model(embedding_sizes=emb_dims, n_numeric=len(num_features_store)).to(DEVICE)
    
    # 定义损失函数和优化器
    criterion = nn.MSELoss() # 使用MSE进行训练以提高稳定性
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5) 
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        for cats, nums, targets in train_loader:
            cats_dev = [c.to(DEVICE) for c in cats]
            nums_dev, targets_dev = nums.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(cats_dev, nums_dev)
            loss = criterion(outputs.squeeze(), targets_dev) 
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            total_train_loss += loss.item()
    
        avg_train_loss_rmse = np.sqrt(total_train_loss / len(train_loader))
        
        # 验证
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for cats, nums, targets in valid_loader:
                cats_dev = [c.to(DEVICE) for c in cats]
                nums_dev, targets_dev = nums.to(DEVICE), targets.to(DEVICE)
                
                outputs = model(cats_dev, nums_dev)
                loss = criterion(outputs.squeeze(), targets_dev) # 直接使用MSE
                total_val_loss += loss.item()
                
        avg_val_loss_rmse = np.sqrt(total_val_loss / len(valid_loader))
        
        print(f"Epoch {epoch+1}/{EPOCHS}, Train RMSE: {avg_train_loss_rmse:.4f}, Valid RMSE: {avg_val_loss_rmse:.4f}")
        
        # 早停逻辑
        if avg_val_loss_rmse < best_val_loss:
            best_val_loss = avg_val_loss_rmse
            torch.save(model.state_dict(), f'model_{store_name}.pth')
            print(f"Model saved for {store_name} with validation RMSE: {best_val_loss:.4f}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    # 加载最佳模型进行预测
    print(f"Loading best model for {store_name} to make predictions.")
    model.load_state_dict(torch.load(f'model_{store_name}.pth'))
    model.eval()
    
    all_predictions = []
    
    valid_indices = valid_df.index
    with torch.no_grad():
        for cats, nums, _ in valid_loader:
            cats_dev = [c.to(DEVICE) for c in cats]
            nums_dev = nums.to(DEVICE)
            preds = model(cats_dev, nums_dev).squeeze().cpu().numpy()
            all_predictions.extend(preds)
    data.loc[valid_indices, 'predictions'] = all_predictions
    

    if not test_df.empty:
        test_dataset = M5Dataset(test_df, cat_features_store, num_features_store, target_col='sold')
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_indices = test_df.index
        all_predictions = []
        with torch.no_grad():
            for cats, nums, _ in test_loader:
                cats_dev = [c.to(DEVICE) for c in cats]
                nums_dev = nums.to(DEVICE)
                preds = model(cats_dev, nums_dev).squeeze().cpu().numpy()
                all_predictions.extend(preds)
        data.loc[test_indices, 'predictions'] = all_predictions
    
    del model, train_loader, valid_loader, train_dataset, valid_dataset
    gc.collect()

if 'id' not in data.columns:
    data.reset_index(inplace=True)
    if 'index' in data.columns and 'id' not in data.columns:
        data.rename(columns={'index': 'id'}, inplace=True)

# 筛选出验证集周期的预测
validation_df = data[(data['d'] >= 1914) & (data['d'] < 1942)][['id', 'd', 'predictions']]
validation_df = pd.pivot_table(validation_df, index='id', columns='d', values='predictions').reset_index()
validation_df.columns = ['id'] + ['F' + str(i + 1) for i in range(28)]
validation_df['id'] = validation_df['id'].astype(str).str.replace('_evaluation', '_validation')

# 筛选出测试集（Evaluation）周期的预测
evaluation_df = data[data['d'] >= 1942][['id', 'd', 'predictions']]
evaluation_df = pd.pivot_table(evaluation_df, index='id', columns='d', values='predictions').reset_index()
evaluation_df.columns = ['id'] + ['F' + str(i + 1) for i in range(28)]

# 合并验证集和评估集的预测
submit = pd.concat([validation_df, evaluation_df]).reset_index(drop=True)

# 保存到CSV
submit.to_csv('submission.csv', index=False)
print(submit.head())

